# HNSW From Scratch

Wiki reference for [Hierarchical Navigable Small World graphs](https://ml-viz-ruby.vercel.app/wiki/hnsw).

> Colab: **File → Save a copy in Drive** before editing.

**The problem it solves.** Exact nearest-neighbour search over a million embeddings scans
every vector — too slow for an interactive request. HNSW is an **approximate** index that
returns *most* of the true neighbours in roughly `O(log N)` hops.

**The core idea.** Build a **skip-list for geometry**: a stack of proximity graphs where the
sparse top layer has long-range links you can teleport across, and each layer down is denser
and refines the position, until layer 0 (all points) does the fine search. A query greedily
hops toward itself, coarse layers first.

**Where it shows up.** The default index in FAISS, Qdrant, Weaviate, pgvector, Milvus, Redis —
the engine behind semantic search and RAG retrieval at scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq
from collections import defaultdict

plt.style.use('dark_background')
C = {'pt': '#64748b', 'q': '#f59e0b', 'path': '#22d3ee', 'hit': '#6366f1', 'edge': '#334155'}
rng = np.random.default_rng(7)

## 1. From scratch

We build HNSW in small pieces. First the two heap-based primitives, then the layer-0 beam
search (`search_layer`), then the greedy descent that stitches the layers together, then
insertion (which reuses search). Distances are squared-Euclidean throughout — monotonic in the
true distance, so rankings are identical and we skip the `sqrt`.

### 1a. The index skeleton and layer assignment

Each node gets a random maximum layer $\ell = \lfloor -\ln(U)\,m_L \rfloor$ with
$m_L = 1/\ln M$. Most points land on layer 0; a few reach higher. Layer 0 holds **all**
points, and uses a larger degree cap $M_0 = 2M$.

In [ ]:
def d2(a, b):
    """Squared Euclidean distance — monotonic in true distance, so rankings match."""
    diff = a - b
    return float(diff @ diff)

class HNSW:
    def __init__(self, M=8, ef_construction=64, seed=0):
        self.M = M                     # max edges per node on layers >= 1
        self.M0 = 2 * M                # max edges on layer 0
        self.ef_construction = ef_construction
        self.m_L = 1.0 / np.log(M)     # level-assignment scale
        self.vectors = []              # id -> np.ndarray
        self.layers = []               # id -> its top layer
        # graph[layer] : dict node_id -> set(neighbour_ids)
        self.graph = defaultdict(lambda: defaultdict(set))
        self.entry_point = None        # id of the single top-layer node
        self.top_layer = -1
        self.rng = np.random.default_rng(seed)

    def _random_level(self):
        u = self.rng.random()
        return int(-np.log(u) * self.m_L)

# We attach the remaining methods in the cells below (HNSW.method = fn) so each
# piece gets its own explanation and stays runnable on its own.

### 1b. `search_layer` — the beam search on one layer

This is the heart of HNSW. It keeps a **candidate frontier** (min-heap, nearest first) and a
**result set** capped at `ef` (kept as a max-heap so the *farthest* is at the top and cheap to
evict). The loop stops early the moment the nearest unexplored candidate is farther than the
current worst result — nothing unexplored can improve the answer.

In [ ]:
def _search_layer(self, q, entry_ids, ef, layer):
    visited = set(entry_ids)
    # candidates: min-heap of (dist, id) — frontier, nearest popped first
    candidates = []
    # results: max-heap of (-dist, id) — worst (farthest) at top for O(1) eviction
    results = []
    for i in entry_ids:
        dist = d2(q, self.vectors[i])
        heapq.heappush(candidates, (dist, i))
        heapq.heappush(results, (-dist, i))
    while candidates:
        c_dist, c = heapq.heappop(candidates)
        worst = -results[0][0]          # current farthest result
        if c_dist > worst:
            break                       # frontier can't improve results
        for e in self.graph[layer][c]:
            if e in visited:
                continue
            visited.add(e)
            e_dist = d2(q, self.vectors[e])
            worst = -results[0][0]
            if e_dist < worst or len(results) < ef:
                heapq.heappush(candidates, (e_dist, e))
                heapq.heappush(results, (-e_dist, e))
                if len(results) > ef:
                    heapq.heappop(results)   # evict the farthest
    return [i for (_, i) in results]

HNSW._search_layer = _search_layer

### 1c. The neighbour-selection heuristic

Naively keeping the `M` *nearest* candidates packs redundant edges into one clump and never
builds the bridges that make the graph navigable. HNSW keeps an edge to a candidate only if it
is closer to the new node than to any already-selected neighbour — favouring **diverse
directions**.

In [ ]:
def _select_neighbours(self, q, candidate_ids, M):
    # sort candidates nearest-first
    cand = sorted(candidate_ids, key=lambda i: d2(q, self.vectors[i]))
    selected = []
    for c in cand:
        if len(selected) >= M:
            break
        dist_cq = d2(q, self.vectors[c])
        # keep c only if it is closer to q than to every already-picked neighbour
        keep = all(dist_cq < d2(self.vectors[c], self.vectors[s]) for s in selected)
        if keep:
            selected.append(c)
    # if the heuristic was too strict, top up with the nearest leftovers
    if len(selected) < M:
        for c in cand:
            if c not in selected:
                selected.append(c)
            if len(selected) >= M:
                break
    return selected

HNSW._select_neighbours = _select_neighbours

### 1d. Insertion — greedy-descend, then connect bottom layers

To insert node $x$ at level $\ell$: greedy-descend (beam width 1) from the entry point down to
layer $\ell+1$, then from $\ell$ down to 0 run `search_layer` with `ef_construction`, pick
neighbours with the heuristic, add **bidirectional** edges, and prune any node that overflows
its degree cap.

In [ ]:
def insert(self, vec):
    vec = np.asarray(vec, dtype=float)
    node = len(self.vectors)
    self.vectors.append(vec)
    level = self._random_level()
    self.layers.append(level)

    if self.entry_point is None:            # first node ever
        self.entry_point = node
        self.top_layer = level
        for lyr in range(level + 1):
            self.graph[lyr][node] = set()
        return node

    ep = [self.entry_point]
    # Phase 1: greedy descent through layers above the new node's level
    for lyr in range(self.top_layer, level, -1):
        ep = self._search_layer(vec, ep, ef=1, layer=lyr)

    # Phase 2: connect from min(level, top) down to 0
    for lyr in range(min(level, self.top_layer), -1, -1):
        cand = self._search_layer(vec, ep, self.ef_construction, lyr)
        M = self.M0 if lyr == 0 else self.M
        neighbours = self._select_neighbours(vec, cand, M)
        self.graph[lyr][node] = set(neighbours)
        for nb in neighbours:
            self.graph[lyr][nb].add(node)
            # prune nb if it now exceeds the degree cap
            cap = self.M0 if lyr == 0 else self.M
            if len(self.graph[lyr][nb]) > cap:
                kept = self._select_neighbours(self.vectors[nb], self.graph[lyr][nb], cap)
                self.graph[lyr][nb] = set(kept)
        ep = cand

    if level > self.top_layer:              # new node tops the hierarchy
        self.entry_point = node
        self.top_layer = level
    return node

HNSW.insert = insert

### 1e. Query — descent, then a wide beam on layer 0

The query mirrors insertion: greedy descent with `ef=1` through the upper layers, then one wide
`search_layer` on the base layer with `ef = max(ef_search, k)`, and return the `k` nearest.

In [ ]:
def search(self, q, k=5, ef_search=32):
    q = np.asarray(q, dtype=float)
    ep = [self.entry_point]
    for lyr in range(self.top_layer, 0, -1):
        ep = self._search_layer(q, ep, ef=1, layer=lyr)
    W = self._search_layer(q, ep, max(ef_search, k), layer=0)
    W.sort(key=lambda i: d2(q, self.vectors[i]))
    return W[:k]

HNSW.search = search

# smoke test on 30 tiny vectors
_ix = HNSW(M=6, ef_construction=32, seed=0)
for _v in rng.standard_normal((30, 4)):
    _ix.insert(_v)
print('methods:', [m for m in ('insert','search','_search_layer','_select_neighbours') ])
print('search returns ids:', _ix.search(rng.standard_normal(4), k=3))

## 2. Cross-check against brute-force exact kNN

There is no tiny stdlib HNSW to compare against, so we validate the *contract*: HNSW should
recover almost all of the true nearest neighbours that an **exact** brute-force scan finds. We
build an index over random vectors and measure **recall@k** — the fraction of the true top-$k$
that HNSW returns — averaged over many queries.

In [ ]:
def exact_knn(data, q, k):
    dists = np.array([d2(q, x) for x in data])
    return set(np.argsort(dists)[:k].tolist())

N, dim, k = 2000, 16, 10
data = rng.standard_normal((N, dim))

index = HNSW(M=8, ef_construction=64, seed=1)
for v in data:
    index.insert(v)

queries = rng.standard_normal((200, dim))

def measure_recall(ef_search):
    hits = 0
    for q in queries:
        approx = set(index.search(q, k=k, ef_search=ef_search))
        truth = exact_knn(data, q, k)
        hits += len(approx & truth)
    return hits / (len(queries) * k)

print(f'N={N}, dim={dim}, k={k}, top layer={index.top_layer}')
for ef in [10, 20, 40, 80, 160]:
    print(f'  ef_search={ef:3d}  ->  recall@{k} = {measure_recall(ef):.3f}')

**What to notice.** Recall climbs toward 1.0 as `ef_search` widens — exactly the recall/latency
knob from the wiki. Even a from-scratch index with `M=8` recovers the large majority of true
neighbours while touching a small fraction of the 2000 points.

## 3. Visualize the search path

In 2-D we can watch a query hop across layer 0 toward its target. We rebuild a small index,
instrument the base-layer search to record which nodes it visits, and draw the graph + the
visited frontier + the true nearest neighbour.

In [ ]:
# small 2-D index we can draw
pts = rng.uniform(0, 10, size=(120, 2))
idx2d = HNSW(M=6, ef_construction=40, seed=3)
for p in pts:
    idx2d.insert(p)

# instrumented layer-0 search that records visited nodes
def traced_search(index, q, k=1, ef_search=16):
    q = np.asarray(q, dtype=float)
    ep = [index.entry_point]
    for lyr in range(index.top_layer, 0, -1):
        ep = index._search_layer(q, ep, ef=1, layer=lyr)
    visited = set(ep)
    candidates = [(d2(q, index.vectors[i]), i) for i in ep]
    heapq.heapify(candidates)
    results = [(-d2(q, index.vectors[i]), i) for i in ep]
    heapq.heapify(results)
    while candidates:
        c_dist, c = heapq.heappop(candidates)
        if c_dist > -results[0][0]:
            break
        for e in index.graph[0][c]:
            if e in visited:
                continue
            visited.add(e)
            e_dist = d2(q, index.vectors[e])
            if e_dist < -results[0][0] or len(results) < ef_search:
                heapq.heappush(candidates, (e_dist, e))
                heapq.heappush(results, (-e_dist, e))
                if len(results) > ef_search:
                    heapq.heappop(results)
    best = min(results, key=lambda r: -r[0])[1]
    return visited, best

q = np.array([5.0, 5.0])
visited, best = traced_search(idx2d, q, ef_search=16)
truth = np.argmin([d2(q, p) for p in pts])
print(f'visited {len(visited)} of {len(pts)} nodes; found {best}, true nearest {truth}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
# layer-0 edges (light)
for node, nbrs in idx2d.graph[0].items():
    for nb in nbrs:
        if nb > node:
            xs = [pts[node, 0], pts[nb, 0]]
            ys = [pts[node, 1], pts[nb, 1]]
            ax.plot(xs, ys, color=C['edge'], lw=0.5, zorder=1)
ax.scatter(pts[:, 0], pts[:, 1], c=C['pt'], s=25, zorder=2, label='points')
vis = np.array(sorted(visited))
ax.scatter(pts[vis, 0], pts[vis, 1], c=C['path'], s=45, zorder=3, label='visited by search')
ax.scatter(*pts[truth], c=C['hit'], s=180, marker='*', zorder=4, label='true nearest')
ax.scatter(*q, c=C['q'], s=160, marker='X', zorder=5, label='query')
ax.set_title('HNSW layer-0 search: query hops toward its neighbour')
ax.legend(loc='upper left'); ax.set_aspect('equal'); plt.tight_layout(); plt.show()

**What to notice.** The teal *visited* nodes trace a short path from the query toward the star.
Only a slice of the graph is ever touched — the small-world edges let the walk converge in a
handful of hops instead of scanning all 120 points.

## 4. Tradeoffs & when to use it

| | HNSW |
|---|---|
| **Recall** | very high, tunable via `ef_search` |
| **Query latency** | very low (~`O(log N)` hops) |
| **Memory** | high — graph edges stored *on top of* the vectors |
| **Build/insert** | slow (~`O(N log N)`); deletes only via tombstone + rebuild |

**Knobs:** `M` (edges/node — recall vs memory), `ef_construction` (build quality vs build time),
`ef_search` (recall vs query latency, the one runtime knob).

**Reach for it** when the corpus is large, read-heavy, and fits in RAM and you want top recall at
low latency. **Look elsewhere** when memory is tight at billion scale (pair with product
quantization / IVF-PQ), the corpus churns constantly (no true delete), or you need exact results
(use a flat index).

**Failure modes:** `ef_search < k` silently caps recall; a low `M` starves recall; a high
delete/update rate degrades the graph until rebuild; metric/normalization mismatch returns
nonsense.

## ✏️ Your turn

### Exercise 1 — recall vs `M`

Build three indexes with `M = 4, 8, 16` (same data, `ef_construction=64`) and print
recall@10 at `ef_search=40` for each. Confirm recall rises with `M`.

In [ ]:
# TODO(you): build indexes for M in [4, 8, 16] over `data` and compare recall.
#
# for M in [4, 8, 16]:
#     ix = HNSW(M=M, ef_construction=64, seed=1)
#     for v in data: ix.insert(v)
#     ... measure recall@10 at ef_search=40 over `queries` ...
#     print(M, recall)

In [ ]:
#@title Solution (run to reveal)
def recall_for(index, ef_search, k=10):
    hits = 0
    for q in queries:
        approx = set(index.search(q, k=k, ef_search=ef_search))
        truth = exact_knn(data, q, k)
        hits += len(approx & truth)
    return hits / (len(queries) * k)

for M in [4, 8, 16]:
    ix = HNSW(M=M, ef_construction=64, seed=1)
    for v in data:
        ix.insert(v)
    print(f'M={M:2d}  recall@10 = {recall_for(ix, 40):.3f}')

### Exercise 2 — count the work

Modify `traced_search` (or add a counter) to report how many distance computations a single
query performs on layer 0, and compare it to the `N` a brute-force scan would do. This is the
sub-linear speedup, made concrete.

In [ ]:
# TODO(you): count distance evaluations for one query and compare to N.

In [ ]:
#@title Solution (run to reveal)
q = np.array([5.0, 5.0])
visited, best = traced_search(idx2d, q, ef_search=16)
# every visited node had its distance to q computed once on layer 0
print(f'distance evals (layer 0): ~{len(visited)}')
print(f'brute force would do:      {len(pts)}')
print(f'speedup:                   ~{len(pts) / max(1, len(visited)):.1f}x on this tiny index')

## Key takeaways

- **HNSW = skip list + navigable small world.** Sparse upper layers teleport; dense layer 0
  refines. Search = greedy descent, then a width-`ef` beam on the base layer.
- **`search_layer` is the whole algorithm** — a frontier heap and a capped result heap with an
  early-exit. Insertion reuses it; the neighbour heuristic keeps the graph navigable.
- **Three knobs:** `M` (memory vs recall), `ef_construction` (build quality), `ef_search` (the
  runtime recall/latency dial).
- **Great for read-heavy in-RAM corpora**; awkward for deletes and billion-scale memory (pair
  with PQ / IVF-PQ there).

**Next:** the [Vector Databases wiki](https://ml-viz-ruby.vercel.app/wiki/vector-databases) for
where HNSW sits among IVF, PQ, and flat indexes, and the
[embeddings lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/03-embeddings-and-semantic-search)
for where the vectors come from.